# Lab 03 · Architecture tools and our own MCP server

**Prepared by Mustafa Assaf** · 5–10 minute example

## Give the program tools through MCP

**So far:** the model only read text that we put into the prompt.

**New:** *tools*, which are functions that read data or take an action. The **Model Context Protocol (MCP)** publishes tools so that any MCP client can discover and call them.

We build a tiny MCP server with two tools. `read_metrics` *reads* project facts. `save_decision` *changes state* by adding a decision to a Python list. The official MCP SDK connects the client and server inside this notebook, so no port, external service, or model key is needed.

## Setup
Run this cell once to load the course helpers, then run the steps below in order. This protocol example needs no model key.

In [ ]:
from importlib.util import find_spec
if not all(find_spec(name) for name in ['coursekit', 'mcp']):
    %pip install -q "agentic-ai-coursekit[mcp] @ https://github.com/mkassaf/agentic-ai-course/archive/aebb52597ab982f0b8b2f0832815b73dd6aca416.zip"

## Step 1 · Build a server with two tools

`@server.tool(...)` publishes an ordinary Python function as a tool. A client sees each tool's `description`, and in Lab 04 a model uses this kind of text to pick a tool. Type hints such as `choice: str` become the tool's argument schema.

In [ ]:
from mcp import Client
from mcp.server import MCPServer
server = MCPServer("Architecture tools")
saved = []  # stands in for a real decision log
@server.tool(description="Read the team size and whether services must deploy independently.")
def read_metrics() -> dict:
    return {"team_size": 3, "independent_deploys": False}
@server.tool(description="Save the chosen architecture.")
def save_decision(choice: str) -> str:
    saved.append(choice)
    return f"Saved: {choice}"

## Step 2 · Discover the tools, then call them

The client first asks which tools exist, then calls them by name with arguments. Here *our code* chooses the order: it reads the facts, applies the policy itself (3 people means modular monolith), and saves that choice. In Lab 04 the model chooses the next step.

A notebook lets you use `await` directly in a cell, and the MCP client needs it.

In [ ]:
async with Client(server) as client:
    listing = await client.list_tools()
    for tool in listing.tools:
        print("Tool:", tool.name, "-", tool.description)
    metrics = await client.call_tool("read_metrics", {})
    print("Read:", metrics.content[0].text)
    result = await client.call_tool("save_decision", {"choice": "modular monolith"})
    print(result.content[0].text)
assert saved == ["modular monolith"], "save_decision should have added one decision."

## What you should see

Two tools with their descriptions, the project facts as JSON, then `Saved: modular monolith`. The check confirms that `save_decision` changed the `saved` list.

## Try it

Add `"service_name": "orders"` to the dictionary that `read_metrics` returns. Run both steps and find the new field in the output. The client needs no change, because it prints whatever the tool returns.

## Takeaway

A *read* is safe to repeat. An *action* changes something, so a real system checks an action before running it. MCP describes and transports tools. It does not choose the architecture or check whether the saved choice is right.

## Read further

- [Official SDK client guide](https://py.sdk.modelcontextprotocol.io/client/)
- [MCP architecture](https://modelcontextprotocol.io/docs/learn/architecture)